In [1]:
from numba.cuda.printimpl import print_item

from Util.Problems import Problem, solution
from Util.BigInt import make_bigint
import Util.math_functions as mathf
import math

class P025(Problem):
    number = 25
    title = "$1000$-digit Fibonacci Number"
    description = """<p>The Fibonacci sequence is defined by the recurrence relation:</p><blockquote>$F_n = F_{n - 1} + F_{n - 2}$, where $F_1 = 1$ and $F_2 = 1$.</blockquote><p>Hence the first $12$ terms will be:</p>
$$\\begin{align}
F_1 &= 1\\\\
F_2 &= 1\\\\
F_3 &= 2\\\\
F_4 &= 3\\\\
F_5 &= 5\\\\
F_6 &= 8\\\\
F_7 &= 13\\\\
F_8 &= 21\\\\
F_9 &= 34\\\\
F_{10} &= 55\\\\
F_{11} &= 89\\\\
F_{12} &= 144
\\end{align}$$
<p>The $12$th term, $F_{12}$, is the first term to contain three digits.</p><p>What is the index of the first term in the Fibonacci sequence to contain $1000$ digits?</p>"""
    digits = 1000

In [2]:
p = P025()
p.describe()

## Problem 25: $1000$-digit Fibonacci Number

<p>The Fibonacci sequence is defined by the recurrence relation:</p><blockquote>$F_n = F_{n - 1} + F_{n - 2}$, where $F_1 = 1$ and $F_2 = 1$.</blockquote><p>Hence the first $12$ terms will be:</p>
$$\begin{align}
F_1 &= 1\\
F_2 &= 1\\
F_3 &= 2\\
F_4 &= 3\\
F_5 &= 5\\
F_6 &= 8\\
F_7 &= 13\\
F_8 &= 21\\
F_9 &= 34\\
F_{10} &= 55\\
F_{11} &= 89\\
F_{12} &= 144
\end{align}$$
<p>The $12$th term, $F_{12}$, is the first term to contain three digits.</p><p>What is the index of the first term in the Fibonacci sequence to contain $1000$ digits?</p>

### Solution notes
We start with a simple brute force solution: calculating all Fibonacci numbers, while tracking their index until one reaches 1000 digits.

In [3]:
@solution(P025, first=True, max_tests= 10, make_fast=False, warmup_args=(P025.digits,))
def brute_force(digits):
    i = 2
    previous_fibonacci = 1
    current_fibonacci = 1
    while current_fibonacci / (10 ** (digits - 1 )) < 1:
        i += 1
        next_fibonacci = previous_fibonacci + current_fibonacci
        previous_fibonacci = current_fibonacci
        current_fibonacci = next_fibonacci
    return i

In [4]:
p.test_once("brute_force")

4782 found after a separate test in 18.906700 ms by brute_force (first)


Using the BigInt class I developed earlier, we can implement the same basic brute force method with numba to make it faster.

In [5]:
@solution(P025, max_tests= 100, make_fast=True, warmup_args=(P025.digits,))
def numba_brute_force(digits):
    i = 2
    previous_fibonacci = make_bigint("1")
    current_fibonacci = make_bigint("1")
    while current_fibonacci.digits() < digits:
        i += 1
        next_fibonacci = previous_fibonacci + current_fibonacci
        previous_fibonacci = current_fibonacci
        current_fibonacci = next_fibonacci
    return i

In [6]:
p.test_once("numba_brute_force")

4782 found after a separate test in 6.197700 ms by numba_brute_force


The function for the $n^{th}$ Fibonacci number $F_n$ is roughly equal to $\frac{\varphi^n}{\sqrt{5}$. The number of digits in a positive integer is equal to $\lfloor{log_{10}{x}}\rfloor + 1$, so in this case:

${log_{10}{\frac{\varphi^n}{\sqrt{5}}} + 1 \\
log_{10}{\varphi^n} - log_{10}{\sqrt{5}} + 1 \\
n \times log_{10}{\varphi} - log_{10}{\sqrt{5}} + 1$

To get a Fibonacci number with 1000 digits, we set this formula equal to 1000 to get:

$\lfloor n \times log_{10}{\varphi} - log_{10}{\sqrt{5}}\rfloor + 1 = 1000$

If we move the 1 to the other side and take the ceiling, this gives us the lowest value for which this is true.

In [7]:
@solution(P025, best= True, make_fast=False, warmup_args=(P025.digits,))
def nth_Fibonacci(digits):
    n = math.ceil((digits - 1 + (math.log(math.sqrt(5)) / math.log(10))) / (math.log(mathf.PHI) / math.log(10)))
    return n

In [8]:
p.test_all()

4782 found after 10 tests in 10.894230 ms by brute_force (first)
4782 found after 1000 tests in 0.000393 ms by nth_Fibonacci (best)
4782 found after 100 tests in 5.679130 ms by numba_brute_force
